# Neuronales Netz - Aktivierungsfunktion, Lernrate, Batch-Size, Epochenanzahl

**systematische Einschränkung der Hyperparameterbreiche**

**ki-308** California House Prising

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LinearRegression

from utils.data import load_and_clean_data, get_train_test_split, Standard_Scaler, Min_Max_Scaler
from utils.evaluation import evaluate_predictions, add_result
from utils.plotting import plot_predicted_vs_actual, plot_residuals, save_fig

plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
def build_model(hidden_layers, activation='relu', learning_rate=0.001):
    """Erstelle ein Sequential-Modell mit gegebener Architektur."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    
    model.add(layers.Dense(1))  # Regression Output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


def train_and_evaluate(model, name, epochs=100, batch_size=32, verbose=0):
    """Trainiere und evaluiere ein Keras-Modell."""
    history = model.fit(
        X_train_nn, y_train_nn,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )
    
    y_train_pred = model.predict(X_train, verbose=0).flatten()
    y_test_pred = model.predict(X_test, verbose=0).flatten()
    
    result = evaluate_predictions(y_train, y_train_pred, y_test, y_test_pred, name)
    add_result(result)
    
    return history, result

## Daten laden und transformieren

In [ ]:
df = load_and_clean_data()

Da wir sehr verschiedene Intervalle haben sollte eine Skalierung sehr sinnvoll sein, um ansonsten z.B. Sigmoid nicht gut funktionieren kann.

In [ ]:
df = load_and_clean_data()
X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train_nn, y_train_nn = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train_nn.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")

In [ ]:
df_standard = Standard_Scaler(df)
X_train_standard, X_test_standard, y_train_standard, y_test_standard, feature_names = get_train_test_split(df_standard)

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard_nn, y_train_standard_nn = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard_nn.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

In [ ]:
df_min0_max1 = Min_Max_Scaler(df, 0, 1)
X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, feature_names = get_train_test_split(df_min0_max1)

val_split = int(0.8 * len(X_train_min0_max1))
X_val, y_val = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_nn, y_train_nn = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_nn.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")

In [ ]:
df_minneg1_max1 = Min_Max_Scaler(df, -1, 1)
X_train_minneg1_max1, X_test_minneg1_max1, y_train_minneg1_max1, y_test_minneg1_max1, feature_names = get_train_test_split(df_minneg1_max1)

val_split = int(0.8 * len(X_train_minneg1_max1))
X_val, y_val = X_train_minneg1_max1[val_split:], y_train_minneg1_max1[val_split:]
X_train_nn, y_train_nn = X_train_minneg1_max1[:val_split], y_train_minneg1_max1[:val_split]

print(f"Training:   {X_train_nn.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test_minneg1_max1.shape}")
print(f"Features:   {feature_names}")

Beginn mit ReLu: (https://www.datacamp.com/de/tutorial/introduction-to-activation-functions-in-neural-networks?dc_referrer=https%3A%2F%2Fwww.google.com%2F)

- rechnerisch kostengünstig auch bei Anwendung mehrerer Schichten
- gängiste Aktivierungsfunktion

Fixparameter:
- Loss: MSE
- Hidden Layers: 2 mit 64 und 32 units (Datensatz zu klein für tieferes Netz und zu geringe Features für breiteres Netz; abnehmende Größen beugen Overfitting vor [Copilot Promt: welche fixparameter wie layer-, unitanzahl etc. sollte man hier wählen?])
- Metric: mae
- learning_rate: 0.001
- Epochen: 100 (von Florian übernommen)

In [ ]:
model_relu_standard = build_model(hidden_layers=[64, 32], activation='relu', learning_rate=0.001)
model_relu_standard.summary()

history_relu_standard, result_relu_standard = train_and_evaluate(model_relu_standard, "NN [64, 32] ReLU", epochs=100)